In [1]:
import cv2
import torch
from ultralytics import YOLO

# ── Setup & Check CUDA ────────────────────────────────────────────────────────
device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load model (YOLOv11m)
model = YOLO("models/weapon_detection_yolov11m.pt")
model.fuse()

# ── Button Configs ───────────────────────────────────────────────────────────
BTN_X, BTN_Y = 20, 20
BTN_W, BTN_H = 180, 45
analyzing = False
mouse_state = {"clicked": False}

def draw_button(frame, active):
    label = "STOP TRACKING" if active else "START TRACKING"
    color = (0, 0, 180) if active else (0, 140, 0) # Red if active, Green if idle
    cv2.rectangle(frame, (BTN_X, BTN_Y), (BTN_X + BTN_W, BTN_Y + BTN_H), color, -1)
    cv2.putText(frame, label, (BTN_X + 15, BTN_Y + 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

def on_mouse(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        if BTN_X <= x <= BTN_X + BTN_W and BTN_Y <= y <= BTN_Y + BTN_H:
            param["clicked"] = True

# ── Initialize Window & Webcam ────────────────────────────────────────────────
cv2.namedWindow("YOLOv11 + ByteTrack")
cv2.setMouseCallback("YOLOv11 + ByteTrack", on_mouse, mouse_state)
cap = cv2.VideoCapture(0)

print("Press 'Q' to quit. Click the button to toggle tracking.")

while True:
    ret, frame = cap.read()
    if not ret: break

    frame = cv2.resize(frame, (640, 480))
    display_frame = frame.copy()

    # ── Handle Button Click ───────────────────────────────────────────────────
    if mouse_state["clicked"]:
        analyzing = not analyzing
        mouse_state["clicked"] = False

    # ── Inference Logic ───────────────────────────────────────────────────────
    if analyzing:
        # persist=True + bytetrack handles the IDs automatically
        results = model.track(
            frame,
            persist=True,
            tracker="bytetrack.yaml",
            conf=0.5,
            device=device,
            verbose=False
        )

        # results[0].plot() includes the Bounding Box + Class + Tracking ID
        display_frame = results[0].plot()
    else:
        cv2.putText(display_frame, "Tracking Paused", (20, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    # ── UI Elements ───────────────────────────────────────────────────────────
    draw_button(display_frame, analyzing)

    cv2.imshow("YOLOv11 + ByteTrack", display_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

CUDA is available!
CUDA Version: 12.4
Number of GPUs: 1
Current GPU Name: NVIDIA GeForce GTX 1650
YOLO11m summary (fused): 126 layers, 20,031,574 parameters, 0 gradients, 67.7 GFLOPs
Press 'q' to quit.


KeyboardInterrupt: 